## Setup

In [ ]:
# Run from the repo root, with HF_TOKEN in .env
import dotenv
import torch

from src.dataset import tissue_dataset
from src.evaluate import evaluate
from src.model import load_base_model, load_peft_model_from_checkpoint, load_processor

dotenv.load_dotenv(".env")

base_model = load_base_model()
processor = load_processor()

## Dataset preprocess

In [ ]:
# Download the Motic human-tissue images into src/dataset first
dataset = tissue_dataset(
    images_dir_path="src/dataset/Motic-Human-tissues",
    split_ratio=0.8,
    train_batch_size=1,
    val_batch_size=8,
)

In [ ]:
print(f"Current type classes: {dataset.type_classes}")
print(f"Current zoom classes: {dataset.zoom_classes}")
print(f"Current focus classes: {dataset.focus_classes}")

In [ ]:
dataset.set_processor(processor)
dataset.build_train_val_loaders()

## Baseline evaluation

In [ ]:
baseline_metrics = evaluate(base_model, dataset, processor)

## Fine tuning

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    # target_modules="all-linear",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],  # Target attention layers only
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

# Apply LoRA configuration to the model
model = get_peft_model(base_model, peft_config)

In [ ]:
print(f"trainable parameters: {model.print_trainable_parameters()}")

In [ ]:
from tqdm import tqdm
from transformers import get_scheduler

epochs = 3
learning_rate = 2e-5
weight_decay = 0.001
warmup_ratio = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
total_steps = len(dataset.train_loader) * epochs
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=int(total_steps * warmup_ratio),
    num_training_steps=total_steps,
)

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataset.train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
    for batch in pbar:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    print(f"Average training loss: {total_loss / len(dataset.train_loader):.4f}")

In [ ]:
evaluate(model, dataset, processor)

In [ ]:
# save the model
from time import strftime

save_path = f"src/models/medgemma-4b-it-lora-{strftime('%Y-%m-%d_%H-%M-%S')}"
model.save_pretrained(save_path)

## Reload the saved adapter

In [ ]:
model = load_peft_model_from_checkpoint(save_path)
val_metrics = evaluate(model, dataset, processor)